<a href="https://colab.research.google.com/github/2403a52026-lgtm/NLP.LabAssignments/blob/main/NLP_LAB_08_2403a52026_B_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import nltk
import re
import math
import random
from collections import Counter, defaultdict


In [3]:
# Sample dataset (replace with your own text file if needed)
with open("corpus.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(text[:500])



Natural language processing is a field of artificial intelligence that focuses on the interaction between computers and human language. It enables machines to read, understand, interpret, and generate text in a meaningful and useful way. Over the years, natural language processing has become an essential component of many modern applications such as search engines, virtual assistants, chatbots, recommendation systems, and machine translation platforms.

Language modeling is one of the core task


In [14]:
def preprocess_text(text):
    text = text.lower()
    sentences = nltk.sent_tokenize(text)

    processed_sentences = []
    for sent in sentences:
        sent = re.sub(r'[^a-z\s]', '', sent)
        words = nltk.word_tokenize(sent)
        words = [word for word in words if word.strip()] # Remove potential empty strings after regex
        if words:
            processed_sentences.append(['<s>'] + words + ['</s>'])

    return processed_sentences

In [16]:
import nltk
nltk.download('punkt')
sentences = preprocess_text(text)
random.shuffle(sentences)

split = int(0.8 * len(sentences))
train_data = sentences[:split]
test_data = sentences[split:]

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [8]:
sentences = preprocess_text(text)
random.shuffle(sentences)

split = int(0.8 * len(sentences))
train_data = sentences[:split]
test_data = sentences[split:]


In [9]:
def build_ngram_model(sentences, n):
    ngrams = []
    for sent in sentences:
        for i in range(len(sent) - n + 1):
            ngrams.append(tuple(sent[i:i+n]))
    return Counter(ngrams)


In [17]:
unigrams = build_ngram_model(train_data, 1)
bigrams = build_ngram_model(train_data, 2)
trigrams = build_ngram_model(train_data, 3)

vocab = set([word for (word,) in unigrams.keys()])
V = len(vocab)

In [11]:
def unigram_prob(word):
    return (unigrams[(word,)] + 1) / (sum(unigrams.values()) + V)

def bigram_prob(w1, w2):
    return (bigrams[(w1, w2)] + 1) / (unigrams[(w1,)] + V)

def trigram_prob(w1, w2, w3):
    return (trigrams[(w1, w2, w3)] + 1) / (bigrams[(w1, w2)] + V)


In [12]:
def sentence_probability(sentence, model="bigram"):
    prob = 1
    for i in range(len(sentence)):
        if model == "unigram":
            prob *= unigram_prob(sentence[i])
        elif model == "bigram" and i > 0:
            prob *= bigram_prob(sentence[i-1], sentence[i])
        elif model == "trigram" and i > 1:
            prob *= trigram_prob(sentence[i-2], sentence[i-1], sentence[i])
    return prob


In [18]:
test_sentences = test_data[:5]

for s in test_sentences:
    print("Sentence:", " ".join(s))
    print("Unigram:", sentence_probability(s, "unigram"))
    print("Bigram:", sentence_probability(s, "bigram"))
    print("Trigram:", sentence_probability(s, "trigram"))
    print()

Sentence: <s> language models play a key role in moderating online platforms and maintaining healthy digital communities </s>
Unigram: 1.9996936497464207e-43
Bigram: 3.3086147773195897e-41
Trigram: 3.03861891031278e-40

Sentence: <s> as the value of n increases the language model captures more context and syntactic structure </s>
Unigram: 4.876327953418508e-40
Bigram: 2.5755915241045104e-42
Trigram: 4.519033871680479e-41

Sentence: <s> acoustic models convert audio signals into candidate word sequences while language models determine which sequence is most likely to be correct </s>
Unigram: 3.243393676971469e-59
Bigram: 2.993755351163721e-58
Trigram: 3.2203113216213965e-57

Sentence: <s> as research progresses language models will become even more powerful enabling smarter more intuitive and more humanlike systems in the future </s>
Unigram: 9.723128668869595e-58
Bigram: 1.849736223490137e-57
Trigram: 3.1941433407501776e-57

Sentence: <s> to address the zero probability problem smoothi

In [19]:
def perplexity(sentence, model="bigram"):
    log_prob = 0
    N = len(sentence)

    for i in range(len(sentence)):
        if model == "unigram":
            log_prob += math.log(unigram_prob(sentence[i]))
        elif model == "bigram" and i > 0:
            log_prob += math.log(bigram_prob(sentence[i-1], sentence[i]))
        elif model == "trigram" and i > 1:
            log_prob += math.log(trigram_prob(sentence[i-2], sentence[i-1], sentence[i]))

    return math.exp(-log_prob / N)


In [20]:
for s in test_sentences:
    print("Sentence:", " ".join(s))
    print("Unigram PP:", perplexity(s, "unigram"))
    print("Bigram PP:", perplexity(s, "bigram"))
    print("Trigram PP:", perplexity(s, "trigram"))
    print()


Sentence: <s> language models play a key role in moderating online platforms and maintaining healthy digital communities </s>
Unigram PP: 324.86882026159054
Bigram PP: 240.5455250866643
Trigram PP: 211.12927515333203

Sentence: <s> as the value of n increases the language model captures more context and syntactic structure </s>
Unigram PP: 205.33601209315515
Bigram PP: 279.5239345406693
Trigram PP: 236.17428648616655

Sentence: <s> acoustic models convert audio signals into candidate word sequences while language models determine which sequence is most likely to be correct </s>
Unigram PP: 349.1403271630218
Bigram PP: 316.98161487575317
Trigram PP: 285.87643258731003

Sentence: <s> as research progresses language models will become even more powerful enabling smarter more intuitive and more humanlike systems in the future </s>
Unigram PP: 301.1556684714455
Bigram PP: 292.85146432127374
Trigram PP: 285.9778635227183

Sentence: <s> to address the zero probability problem smoothing techni